# Chuẩn bị đặc trưng RegionCLIP cho XE 10k/1k
Chọn CPU, bật Internet và secret codex có quyền đọc v146/v149. Gắn csa-prompt-full-karpathy-v175. Giữ thư mục regionclip_xe10k_inputs.


In [ ]:
EXPECTED_MANIFEST='a5dd6f8ee5ef933e540b0c4a31d3a6c21924c74db71cf831894fd2780d2b1be7'
EXPECTED_TEXT={'checkpoint_sha256': 'afbc58d7d4bd96e2b10cc0d1928a912ed519c3b3215df7e5f3285e4bb09c7c58', 'prompt_links_sha256': 'e90c79b3b04540630b4bd2d279a4070ef05da7f4b65970c952851c37e2423bf0', 'shape': [80, 1024], 'texts_sha256': '717c9dfb79efb8fa9f000ac6c80ae201097b867b28f5c8946505ada0c4c70992', 'embeddings_sha256': '1d549de3016c662281b853da695d4e378042e162ca81154826a9d387969884b5'}

from pathlib import Path
import hashlib,json,os,subprocess,sys,tempfile
import numpy as np

SECRET_NAME='codex'
V149='ducanh3/notebook7bc83c4c76/149'
V146='ducanh3/notebook7bc83c4c76/146'
PROMPT_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-prompt-full-karpathy-v175')
OUT=Path('/kaggle/working/regionclip_xe10k_inputs')
download=Path(tempfile.mkdtemp(prefix='regionclip-inputs-'))
from kaggle_secrets import UserSecretsClient
env=dict(os.environ)
for key in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):
    env.pop(key,None)
env['KAGGLE_API_TOKEN']=UserSecretsClient().get_secret(SECRET_NAME).strip()
cli=download/'cli'
subprocess.run([sys.executable,'-m','pip','install','-q','--target',str(cli),'kaggle==2.2.5'],
               check=True,timeout=300)
env['PYTHONPATH']=str(cli)

def fetch(version,pattern):
    dest=download/version.rsplit('/',1)[-1]
    dest.mkdir()
    subprocess.run([sys.executable,'-c','from kaggle.cli import main; main()','kernels','output',
                    version,'-p',str(dest),'--file-pattern',pattern],env=env,check=True,timeout=900)
    return dest

def sha(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream,'sha256').hexdigest()

def checked(path,pin):
    if sha(path)!=pin:
        raise ValueError(f'Changed file: {path.name}')
    return path

a=fetch(V149,r'(^|/)regionclip_slot_cache10k/(result[.]json|cohort[.]json|shard_[0-9]{3}[.](json|npz))$')
b=fetch(V146,r'(^|/)regionclip_object_text_v2/(object_texts[.]json|object_text_embeddings[.]npy)$')
manifest_path,=(x for x in a.rglob('result.json') if sha(x)==EXPECTED_MANIFEST)
root=manifest_path.parent
manifest=json.loads(manifest_path.read_bytes())
cohort=json.loads(checked(root/'cohort.json',manifest['cohort_sha256']).read_bytes())
ids=cohort['train']+cohort['val']
if len(set(ids))!=11000 or [len(cohort[k]) for k in ('train','val')]!=[10000,1000]:
    raise ValueError('Wrong cohort')
index={k:i for i,k in enumerate(ids)}
text_path,=b.rglob('object_texts.json')
texts=json.loads(checked(text_path,EXPECTED_TEXT['texts_sha256']).read_bytes())
text_vectors=np.load(checked(text_path.with_name('object_text_embeddings.npy'),
                           EXPECTED_TEXT['embeddings_sha256']),allow_pickle=False)
if text_vectors.shape!=(80,1024) or text_vectors.dtype!=np.float32 or not np.isfinite(text_vectors).all():
    raise ValueError('Wrong text vectors')
lookup={t:i for i,t in enumerate(texts)}
if len(lookup)!=80:
    raise ValueError('Wrong prompt vocabulary')
links_path,=PROMPT_ROOT.rglob('prompt_links.jsonl')
checked(links_path,EXPECTED_TEXT['prompt_links_sha256'])
links={}
with links_path.open(encoding='utf8') as stream:
    for row in map(json.loads,stream):
        k=row['coco_id']
        if k in index:
            if k in links or row['group']!=('train' if index[k]<10000 else 'val'):
                raise ValueError('Duplicate or wrong-split link')
            links[k]=row
if set(links)!=set(index):
    raise ValueError('Incomplete prompt links')

OUT.mkdir(exist_ok=False)
region=np.lib.format.open_memmap(OUT/'region_vectors.npy',mode='w+',dtype=np.float32,shape=(11000,8,1024))
text_id=np.full((11000,8),-1,np.int16)
boxes=np.zeros((11000,8,4),np.float32)
valid=np.zeros((11000,8),np.bool_)
seen=set()
for shard in manifest['shards']:
    rows=json.loads(checked(root/shard['metadata'],shard['metadata_sha256']).read_bytes())
    if len(rows)!=shard['images']:
        raise ValueError('Wrong shard size')
    with np.load(checked(root/shard['vectors'],shard['vectors_sha256']),allow_pickle=False) as vectors:
        if set(vectors.files)!={str(r['coco_id']) for r in rows}:
            raise ValueError('Wrong vector coverage')
        for r in rows:
            cid=r['coco_id']
            if cid in seen or cid not in index:
                raise ValueError('Unexpected/duplicate cache ID')
            seen.add(cid)
            j=index[cid]
            link=links[cid]
            slots=[dict(instance_id=s['instance_id'],roi=s['roi'],bbox=s['roi_bbox'],text=s['text']) for s in link['objects']]
            if slots!=r['slots'] or r['group']!=link['group'] or r['filename']!=link['filename']:
                raise ValueError(f'Changed prompt–ROI slot: {cid}')
            a=vectors[str(cid)]
            count=len(slots)
            if count>8 or a.shape!=(count,1024) or a.dtype!=np.float32 or not np.isfinite(a).all():
                raise ValueError('Malformed region features')
            if count:
                region[j,:count]=a
                boxes[j,:count]=[s['bbox'] for s in slots]
                text_id[j,:count]=[lookup[s['text']] for s in slots]
                valid[j,:count]=True
    print('Checked',shard['vectors'],flush=True)
region.flush()
if seen!=set(index) or int(valid.sum())!=32832 or int((~valid.any(1)).sum())!=286:
    raise ValueError('Coverage mismatch')
for name,array in dict(coco_ids=np.asarray(ids),boxes=boxes,text_id=text_id,
                       valid=valid,text_vectors=text_vectors).items():
    np.save(OUT/(name+'.npy'),array,allow_pickle=False)
summary=dict(status='COMPLETE',images=manifest['images'],slots=int(valid.sum()),empty_images=int((~valid.any(1)).sum()),
             versions=[V149,V146],files_sha256={f.name:sha(f) for f in sorted(OUT.glob('*.npy'))})
(OUT/'result.json').write_text(json.dumps(summary,indent=2),encoding='utf8')
print(json.dumps(summary,indent=2))
